# HR Policy Knowledge Base — Corpus Exploration (EDA)

This is **not** a classical tabular-ML EDA notebook — this project has no tabular dataset to profile. What matters for a RAG system is the *document corpus* the chatbot retrieves from, so this notebook explores that instead:

- how many PDFs / pages are in the knowledge base
- how much text was extracted per page (catches silently-broken/scanned PDFs early)
- how chunking behaves on the real corpus (chunk count, chunk length distribution)
- a few sample chunks, to eyeball retrieval quality before wiring up the LLM

Run this **after** adding your PDFs to `data/policies/`, and ideally before your first `python ingest.py` — if a PDF shows 0 extracted pages here, ingestion will skip it too, and it's easier to catch that here than to debug a chatbot that mysteriously can't answer from one of your documents.


In [2]:
import sys
from pathlib import Path

# So the notebook can import the `app` package and `config` module when run
# from a `notebooks/` subfolder as well as from the project root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists() and (PROJECT_ROOT.parent / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from app.rag_pipeline import discover_pdfs, extract_pages_from_pdf, build_documents
from config import settings

print(f"Looking for PDFs in: {settings.POLICIES_DIR}")


ModuleNotFoundError: No module named 'fitz'

## 1. Discover PDFs


In [ ]:
pdf_paths = discover_pdfs()

if not pdf_paths:
    print(
        "No PDFs found yet. Add your HR policy / labour law PDFs to "
        f"'{settings.POLICIES_DIR}' and re-run this cell."
    )
else:
    print(f"Found {len(pdf_paths)} PDF(s):")
    for p in pdf_paths:
        print(f"  - {p.name}")


## 2. Extract pages and inspect text quality


In [ ]:
all_pages = []
for pdf_path in pdf_paths:
    all_pages.extend(extract_pages_from_pdf(pdf_path))

page_rows = [
    {
        "source_file": page.source_file,
        "page_number": page.page_number,
        "char_count": len(page.text),
        "word_count": len(page.text.split()),
    }
    for page in all_pages
]
df_pages = pd.DataFrame(page_rows)
df_pages


In [ ]:
# Pages per source file — a PDF with far fewer non-empty pages than its
# actual page count usually means it's a scanned image PDF with no OCR
# text layer, which PyMuPDF can't extract from.
df_pages.groupby("source_file")["page_number"].count().rename("pages_with_text")


In [ ]:
if not df_pages.empty:
    df_pages["word_count"].plot(
        kind="hist", bins=20, title="Word count per page", xlabel="words"
    )
    plt.show()
else:
    print("No pages to plot yet — add PDFs first.")


## 3. Chunking — same logic `ingest.py` uses

This calls the *exact same* `build_documents` function the production ingestion pipeline (`app/rag_pipeline.py`) uses, so what you see here is exactly what will get embedded and stored — not an approximation.


In [ ]:
documents = build_documents(all_pages)

chunk_rows = [
    {
        "source_file": doc.metadata["source_file"],
        "page_number": doc.metadata["page_number"],
        "chunk_index": doc.metadata["chunk_index"],
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
    }
    for doc in documents
]
df_chunks = pd.DataFrame(chunk_rows)
print(f"Total chunks: {len(df_chunks)}")
df_chunks.describe()


In [ ]:
if not df_chunks.empty:
    df_chunks["word_count"].plot(
        kind="hist", bins=20, title="Word count per chunk", xlabel="words"
    )
    plt.show()

    df_chunks.groupby("source_file").size().plot(
        kind="barh", title="Chunks per source document", xlabel="chunk count"
    )
    plt.show()
else:
    print("No chunks to plot yet — add PDFs first.")


## 4. Eyeball a few sample chunks

Quick sanity check before wiring up retrieval: do these chunks actually read as coherent policy text, or did PDF extraction mangle formatting (tables, multi-column layouts, etc. are common culprits)?


In [ ]:
for doc in documents[:3]:
    print(f"--- {doc.metadata['source_file']}, p.{doc.metadata['page_number']} "
          f"(chunk {doc.metadata['chunk_index']}) ---")
    print(doc.page_content[:400])
    print()


## 5. Tuning signal for `config.py` / `.env`

If most chunks are far smaller than `CHUNK_SIZE` (1000 chars by default), your PDFs likely have short paragraphs/lots of line breaks — consider lowering `CHUNK_SIZE` so retrieval doesn't waste context window on padding. If chunks are consistently hitting the max size, unrelated content may be getting concatenated — consider adding more separators or lowering `CHUNK_SIZE` for tighter topic boundaries.


In [ ]:
if not df_chunks.empty:
    print(f"Configured CHUNK_SIZE: {settings.CHUNK_SIZE} chars")
    print(f"Actual mean chunk length: {df_chunks['char_count'].mean():.0f} chars")
    print(f"Actual max chunk length:  {df_chunks['char_count'].max()} chars")
